## Goal of NB 1
- connect to TMDB (The Movie Database) API
- collect movie data
- save it into a pandas DataFrame
- export it as CSV for the next notebook

## Import the libraries

In [1]:
import requests
import pandas as pd
import time

## Add API key

In [2]:
API_KEY = "4db017d7f69398f91238d85a3fab3d43"
BASE_URL = "https://api.themoviedb.org/3"

## Test the API connection

In [3]:
url = f"{BASE_URL}/movie/popular?api_key={API_KEY}&language=en-US&page=1"
response = requests.get(url)

print("Status code:",  response.status_code)
data = response.json()
print(data.keys())

Status code: 200
dict_keys(['page', 'results', 'total_pages', 'total_results'])


## (4) Check one movie sample

In [ ]:
movies = data['results']
movies[0]

{'adult': False,
 'backdrop_path': '/6yeVcxFR0j08vlv2OlL6zbewm4D.jpg',
 'genre_ids': [28, 878, 53],
 'id': 1265609,
 'original_language': 'en',
 'original_title': 'War Machine',
 'overview': 'On one last grueling mission during Army Ranger training, a combat engineer must lead his unit in a fight against a giant otherworldly killing machine.',
 'popularity': 374.3784,
 'poster_path': '/tlPgDzwIE7VYYIIAGCTUOnN4wI1.jpg',
 'release_date': '2026-02-12',
 'title': 'War Machine',
 'video': False,
 'vote_average': 7.256,
 'vote_count': 1080}

## (5) Create a function to fetch popular movies

In [5]:
def fetch_popular_movies(api_key, total_pages=5):
    all_movies = []
    
    for page in range(1, total_pages + 1):
        url = f"{BASE_URL}/movie/popular?api_key={api_key}&language=en-US&page={page}"
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            results = data.get("results", [])
            
            for movie in results:
                all_movies.append({
                    "id": movie.get("id"),
                    "title": movie.get("title"),
                    "overview": movie.get("overview"),
                    "genre_ids": movie.get("genre_ids"),
                    "release_date": movie.get("release_date"),
                    "vote_average": movie.get("vote_average"),
                    "vote_count": movie.get("vote_count"),
                    "popularity": movie.get("popularity"),
                    "original_language": movie.get("original_language")
                })
            
            print(f"Page {page} collected successfully.")
        else:
            print(f"Failed on page {page}. Status code: {response.status_code}")
        
        time.sleep(0.5)  # polite delay
    
    return pd.DataFrame(all_movies)

## (6) Run the function

In [6]:
movies_df = fetch_popular_movies(API_KEY, total_pages=10)
movies_df.head()

Page 1 collected successfully.
Page 2 collected successfully.
Page 3 collected successfully.
Page 4 collected successfully.
Page 5 collected successfully.
Page 6 collected successfully.
Page 7 collected successfully.
Page 8 collected successfully.
Page 9 collected successfully.
Page 10 collected successfully.


,id,title,overview,genre_ids,release_date,vote_average,vote_count,popularity,original_language
0,1265609,War Machine,On one last grueling mission during Army Range...,"[28, 878, 53]",2026-02-12,7.256,1080,374.3784,en
1,875828,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,"[80, 18]",2026-03-05,7.400,298,370.3607,en
2,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]",2026-03-15,8.251,309,323.2718,en
3,1290821,Shelter,A man living in self-imposed exile on a remote...,"[28, 80, 53]",2026-01-28,6.600,359,323.2197,en
4,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,"[878, 12, 14]",2025-12-17,7.265,1913,319.3325,en


## (7) Check dataset shape and info

In [7]:
print("Shape:", movies_df.shape)
movies_df.info()

Shape: (200, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 200 non-null    int64  
 1   title              200 non-null    object 
 2   overview           200 non-null    object 
 3   genre_ids          200 non-null    object 
 4   release_date       200 non-null    object 
 5   vote_average       200 non-null    float64
 6   vote_count         200 non-null    int64  
 7   popularity         200 non-null    float64
 8   original_language  200 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 14.2+ KB


## (8) Get Genre Mapping

In [8]:
genre_url = f"{BASE_URL}/genre/movie/list?api_key={API_KEY}&language=en-US"
genre_response = requests.get(genre_url)

genre_data = genre_response.json()
genre_data

{'genres': [{'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Adventure'},
  {'id': 16, 'name': 'Animation'},
  {'id': 35, 'name': 'Comedy'},
  {'id': 80, 'name': 'Crime'},
  {'id': 99, 'name': 'Documentary'},
  {'id': 18, 'name': 'Drama'},
  {'id': 10751, 'name': 'Family'},
  {'id': 14, 'name': 'Fantasy'},
  {'id': 36, 'name': 'History'},
  {'id': 27, 'name': 'Horror'},
  {'id': 10402, 'name': 'Music'},
  {'id': 9648, 'name': 'Mystery'},
  {'id': 10749, 'name': 'Romance'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 10770, 'name': 'TV Movie'},
  {'id': 53, 'name': 'Thriller'},
  {'id': 10752, 'name': 'War'},
  {'id': 37, 'name': 'Western'}]}

## (9) Create genre dictionary

In [9]:
genre_dict = {}

for genre in genre_data["genres"]:
    genre_dict[genre["id"]] = genre["name"]
genre_dict

{28: 'Action',
 12: 'Adventure',
 16: 'Animation',
 35: 'Comedy',
 80: 'Crime',
 99: 'Documentary',
 18: 'Drama',
 10751: 'Family',
 14: 'Fantasy',
 36: 'History',
 27: 'Horror',
 10402: 'Music',
 9648: 'Mystery',
 10749: 'Romance',
 878: 'Science Fiction',
 10770: 'TV Movie',
 53: 'Thriller',
 10752: 'War',
 37: 'Western'}

## (10) Convert genre IDs to names

In [10]:
def convert_genre_ids_to_names(genre_ids, genre_dict) :
    if isinstance(genre_ids, list):
        return [genre_dict.get(gid, "Unknown") for gid in genre_ids]
        

In [12]:
movies_df["genres"] = movies_df["genre_ids"].apply(lambda x: convert_genre_ids_to_names(x, genre_dict))
movies_df[["title", "genre_ids", "genres"]].head()

,title,genre_ids,genres
0,War Machine,"[28, 878, 53]","[Action, Science Fiction, Thriller]"
1,Peaky Blinders: The Immortal Man,"[80, 18]","[Crime, Drama]"
2,Project Hail Mary,"[878, 12]","[Science Fiction, Adventure]"
3,Shelter,"[28, 80, 53]","[Action, Crime, Thriller]"
4,Avatar: Fire and Ash,"[878, 12, 14]","[Science Fiction, Adventure, Fantasy]"


## (11) Clean the data set

In [13]:
movies_df = movies_df[[
    "id",
    "title",
    "overview",
    "genres",
    "release_date",
    "vote_average",
    "vote_count",
    "popularity",
    "original_language"
]]

## (12) Check missing values

In [14]:
movies_df.isnull().sum()

id                   0
title                0
overview             0
genres               0
release_date         0
vote_average         0
vote_count           0
popularity           0
original_language    0
dtype: int64

## (13) Save to CSV

In [15]:
movies_df.to_csv("movies_dataset.csv", index=False)
print("Dataset saved successfully")

Dataset saved successfully


## (14) Preview final dataset

In [16]:
movies_df.head(10)

,id,title,overview,genres,release_date,vote_average,vote_count,popularity,original_language
0,1265609,War Machine,On one last grueling mission during Army Range...,"[Action, Science Fiction, Thriller]",2026-02-12,7.256,1080,374.3784,en
1,875828,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,"[Crime, Drama]",2026-03-05,7.400,298,370.3607,en
2,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[Science Fiction, Adventure]",2026-03-15,8.251,309,323.2718,en
3,1290821,Shelter,A man living in self-imposed exile on a remote...,"[Action, Crime, Thriller]",2026-01-28,6.600,359,323.2197,en
4,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,"[Science Fiction, Adventure, Fantasy]",2025-12-17,7.265,1913,319.3325,en
5,1159559,Scream 7,When a new Ghostface killer emerges in the qui...,"[Horror, Mystery, Crime]",2026-02-25,5.856,420,326.0984,en
6,1171145,Crime 101,When an elusive thief whose high-stakes heists...,"[Crime, Thriller]",2026-02-11,7.023,200,286.3454,en
7,840464,Greenland 2: Migration,Having found the safety of the Greenland bunke...,"[Adventure, Thriller, Science Fiction]",2026-01-07,6.386,653,205.8209,en
8,1523145,Your Heart Will Be Broken,High school student Polina is saved from bully...,"[Romance, Drama]",2026-03-26,6.300,3,228.1928,ru
9,1084242,Zootopia 2,After cracking the biggest case in Zootopia's ...,"[Animation, Comedy, Adventure, Family, Mystery]",2025-11-26,7.610,2284,179.3031,en
